# Pseudo-Bulk Deconvolution Using the Fusion Signature

Uses the compact gene signature and RF classifier from `Fusion_embedding_classifier_comparison.ipynb`
(Sections 11-13) to build and test a bulk deconvolution pipeline:

1. Split single cells into a **reference pool** (build per-class average expression profiles) and a
   **bulk-simulation pool** (drawn from to build synthetic pseudo-bulk mixtures) -- kept separate so
   deconvolution is never evaluated against the same cells used to build its own reference.
2. Simulate pseudo-bulk samples with random, known mixing fractions of fused/parental/resistant
   cells, combined by averaging.
3. Deconvolve each pseudo-bulk with non-negative least squares (NNLS) against the reference matrix
   to estimate mixing fractions.
4. Cross-check against an independent estimate: run the single-cell RF classifier on the individual
   cells that went into each pseudo-bulk and take the empirical fraction of each predicted label.
5. Compare both estimates against the known true fractions, and against each other.

Needs `fusion_signature_expression.h5ad` and `fusion_signature_classifier.pkl`, both written by the
last cell of Section 13 in `Fusion_embedding_classifier_comparison.ipynb` -- run that notebook
through Section 13 first if these files aren't present yet.

## Imports

In [ ]:
import numpy as np
import pandas as pd
import anndata as ad
import matplotlib.pyplot as plt
import joblib

from sklearn.model_selection import train_test_split
from sklearn.feature_selection import f_classif
from sklearn.linear_model import LinearRegression, ElasticNet
from sklearn.ensemble import RandomForestRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from scipy.optimize import nnls
from scipy.stats import pearsonr

## 1. Load the Fusion Signature Expression and Classifier

In [ ]:
SIGNATURE_EXPRESSION_PATH = 'fusion_signature_expression.h5ad'
SIGNATURE_CLASSIFIER_PATH = 'fusion_signature_classifier.pkl'

sig_adata = ad.read_h5ad(SIGNATURE_EXPRESSION_PATH)
clf_rf_sig = joblib.load(SIGNATURE_CLASSIFIER_PATH)

signature_genes = sig_adata.var_names.tolist()
print(f'{len(signature_genes)} signature genes, {sig_adata.n_obs} cells')
print('Classes:', sorted(sig_adata.obs["sample"].unique()))

## 2. Simulation Parameters

In [ ]:
RANDOM_SEED = 42
REF_POOL_FRACTION = 0.5   # fraction of cells reserved to build per-class reference profiles
N_PSEUDOBULKS = 200
N_CELLS_PER_BULK = 200
DIRICHLET_ALPHA = 1.0     # uniform-over-simplex sampling of mixing fractions across pseudobulks
REGRESSION_TEST_FRACTION = 0.3   # held-out pseudo-bulks for evaluating the ElasticNet / RF regressors
BIC_LOG_EPS = 1e-6                # floor added before log-transforming fractions for regression targets

rng = np.random.default_rng(RANDOM_SEED)
classes = sorted(sig_adata.obs['sample'].unique())
print('Classes (fixed order used throughout):', classes)

## 3. Split Cells into a Reference Pool and a Bulk-Simulation Pool

Stratified by `sample` so both pools stay representative of all three classes. The reference pool
builds the per-class signature matrix used by NNLS; only the bulk-simulation pool is ever sampled
into pseudo-bulks, so deconvolution is never evaluated against its own reference cells.

In [ ]:
ref_idx, bulk_idx = train_test_split(
    np.arange(sig_adata.n_obs),
    test_size=1 - REF_POOL_FRACTION,
    random_state=RANDOM_SEED,
    stratify=sig_adata.obs['sample'].values,
)
print(f'Reference pool: {len(ref_idx)} cells | Bulk-simulation pool: {len(bulk_idx)} cells')

bulk_pool_class_idx = {
    c: bulk_idx[sig_adata.obs['sample'].values[bulk_idx] == c]
    for c in classes
}
for c in classes:
    print(f'  {c}: {len(bulk_pool_class_idx[c])} cells available in the bulk-simulation pool')

## 4. Build the Reference Signature Matrix

NNLS assumes linear mixing (bulk = weighted sum of per-cell profiles), but the exported expression
is `asinh`-transformed -- `asinh` isn't additive across cells (`asinh(mean(x)) != mean(asinh(x))`),
so we invert it back to the linear (normalized) scale with `sinh` before averaging. This only
affects the reference matrix and pseudo-bulk profiles built below; the single-cell classifier keeps
using the original `asinh`-space values it was trained on.

In [ ]:
def to_linear(X):
    return np.sinh(X)

expr_asinh = sig_adata.X
if hasattr(expr_asinh, 'toarray'):
    expr_asinh = expr_asinh.toarray()
expr_linear = to_linear(expr_asinh)

sample_labels = sig_adata.obs['sample'].values
reference_matrix = np.column_stack([
    expr_linear[ref_idx][sample_labels[ref_idx] == c].mean(axis=0)
    for c in classes
])  # (n_genes, n_classes)
print('Reference matrix shape:', reference_matrix.shape)

## 5. Simulate Pseudo-Bulk Mixtures

For each of `N_PSEUDOBULKS` samples: draw a random composition from a symmetric Dirichlet
(`DIRICHLET_ALPHA`), convert it to exact per-class cell counts via a multinomial draw (so counts
always sum to `N_CELLS_PER_BULK`), then randomly draw that many cells **with replacement** from each
class's bulk-simulation pool (bootstrap-style -- with `N_PSEUDOBULKS x N_CELLS_PER_BULK` total draws
spread over three class pools of a few thousand cells each, sampling without replacement would risk
exhausting a pool partway through). The true fraction recorded for each pseudo-bulk is the actual
realized count-based composition (`counts / N_CELLS_PER_BULK`), not the raw Dirichlet draw.

In [ ]:
pseudobulk_records = []

for b in range(N_PSEUDOBULKS):
    true_props = rng.dirichlet(np.full(len(classes), DIRICHLET_ALPHA))
    counts = rng.multinomial(N_CELLS_PER_BULK, true_props)
    true_fractions = counts / N_CELLS_PER_BULK

    drawn_indices = []
    for c, n_c in zip(classes, counts):
        if n_c == 0:
            continue
        pool = bulk_pool_class_idx[c]
        drawn_indices.append(rng.choice(pool, size=n_c, replace=True))
    drawn_indices = np.concatenate(drawn_indices) if drawn_indices else np.array([], dtype=int)

    bulk_profile_linear = expr_linear[drawn_indices].mean(axis=0)

    pseudobulk_records.append({
        'bulk_id': b,
        'true_fractions': true_fractions,
        'cell_indices': drawn_indices,
        'bulk_profile_linear': bulk_profile_linear,
    })

print(f'Simulated {len(pseudobulk_records)} pseudo-bulks of {N_CELLS_PER_BULK} cells each')

## 6. Select Signature Length (k) via BIC

Rank signature genes by how well they separate the three classes (ANOVA F-statistic on the reference
pool), then sweep the number of top-ranked genes `k` used downstream. For each `k`, fit a multivariate
linear regression of `log(true_fraction + eps)` on the `k`-gene pseudo-bulk profile and score it with
BIC (`n * log|Σ_hat| + (k+1)*q * log(n)`, treating each class's slopes+intercept as free parameters and
scoring the joint residual covariance rather than 3 independent BICs). The `k` minimizing BIC is used to
subset `reference_matrix` for everything below (NNLS and the new regression models). The pretrained
single-cell classifier (`clf_rf_sig`) is unaffected — it's a fixed artifact trained on the full original
panel and can't be resized without retraining.

`k` is capped well below `N_PSEUDOBULKS` so the per-`k` regression stays comfortably overdetermined;
otherwise the residual covariance approaches singular and BIC's log-determinant term blows up, spuriously
favoring very large `k`.

In [ ]:
f_scores, _ = f_classif(expr_asinh[ref_idx], sample_labels[ref_idx])
gene_rank = np.argsort(f_scores)[::-1]  # most discriminative genes first

true_frac_matrix = np.array([rec['true_fractions'] for rec in pseudobulk_records])
bulk_profile_matrix = np.array([rec['bulk_profile_linear'] for rec in pseudobulk_records])
log_frac_matrix = np.log(true_frac_matrix + BIC_LOG_EPS)

n_bulks, n_classes = log_frac_matrix.shape
k_max = min(len(signature_genes), max(5, n_bulks // (2 * n_classes)))
k_grid = sorted(set(np.round(np.geomspace(5, k_max, 15)).astype(int)))

bic_records = []
for k in k_grid:
    top_k_idx = gene_rank[:k]
    X_k = bulk_profile_matrix[:, top_k_idx]
    reg = LinearRegression().fit(X_k, log_frac_matrix)
    residuals = log_frac_matrix - reg.predict(X_k)
    sigma_hat = (residuals.T @ residuals) / n_bulks
    sign, logdet = np.linalg.slogdet(sigma_hat)
    if sign <= 0:
        continue  # near-singular residual covariance at this k - skip, not a meaningful fit
    n_params = (k + 1) * n_classes  # per-class slopes + intercept
    bic = n_bulks * logdet + n_params * np.log(n_bulks)
    bic_records.append({'k': k, 'bic': bic})

bic_df = pd.DataFrame(bic_records)
best_k = int(bic_df.loc[bic_df['bic'].idxmin(), 'k'])
print(f'BIC-selected signature length: k = {best_k} genes (of {len(signature_genes)} candidates, search capped at k_max={k_max})')

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(bic_df['k'], bic_df['bic'], marker='o')
ax.axvline(best_k, color='crimson', linestyle='--', label=f'k* = {best_k}')
ax.set_xlabel('Signature length (k genes)')
ax.set_ylabel('BIC (multivariate log-fraction regression)')
ax.set_title('Choosing signature length by BIC')
ax.legend()
plt.tight_layout()
plt.savefig('signature_length_bic.png')
plt.show()

top_k_idx = gene_rank[:best_k]
signature_genes_k = [signature_genes[i] for i in top_k_idx]
reference_matrix = reference_matrix[top_k_idx, :]
print(f'Reference matrix reduced to selected signature: {reference_matrix.shape}')

## 7. Deconvolve Each Pseudo-Bulk with NNLS

In [ ]:
for rec in pseudobulk_records:
    coeffs, _residual = nnls(reference_matrix, rec['bulk_profile_linear'][top_k_idx])
    total = coeffs.sum()
    rec['nnls_fractions'] = coeffs / total if total > 0 else np.zeros_like(coeffs)

print('Example (first pseudo-bulk):')
print('  true:', dict(zip(classes, pseudobulk_records[0]['true_fractions'].round(3))))
print('  nnls:', dict(zip(classes, pseudobulk_records[0]['nnls_fractions'].round(3))))

## 8. Cross-Check: Aggregate Single-Cell Classifier Predictions per Pseudo-Bulk

Runs `clf_rf_sig` (the RF-only fusion-signature classifier) on the individual cells drawn into each
pseudo-bulk, then takes the empirical fraction of each predicted label -- an independent estimate of
composition that doesn't go through NNLS or the reference matrix at all.

In [ ]:
for rec in pseudobulk_records:
    cell_expr_asinh = expr_asinh[rec['cell_indices']]
    preds = clf_rf_sig.predict(cell_expr_asinh)
    counts = pd.Series(preds).value_counts()
    rec['sc_fractions'] = np.array([counts.get(c, 0) for c in classes]) / len(preds)

print('Example (first pseudo-bulk):')
print('  true:', dict(zip(classes, pseudobulk_records[0]['true_fractions'].round(3))))
print('  single-cell:', dict(zip(classes, pseudobulk_records[0]['sc_fractions'].round(3))))

## 9. Direct Regression Deconvolution (ElasticNet / RF, log-ratio + softmax)

Instead of solving a signature-matrix regression (NNLS) or aggregating per-cell predictions, fit a model
directly on the simulated pseudo-bulks: `bulk expression -> composition`. Raw multi-output regression
predictions aren't constrained to be non-negative or to sum to 1, so rather than regressing on the
fractions themselves, each model is trained on `log(true_fraction + eps)` and its raw 3-output prediction
is passed back through a softmax at inference time - since `softmax(log(p)) = p`, this is the correct
inverse transform for compositional (simplex-valued) targets, not just an ad hoc clamp. Features are the
`arcsinh`-transformed pseudo-bulk profile restricted to the BIC-selected `top_k_idx` genes.

Trained/evaluated with a train/test split across pseudo-bulks (unlike NNLS and the single-cell classifier,
which are applied to every pseudo-bulk directly - a fitted regressor evaluated on its own training data
would be optimistic).

In [ ]:
def softmax(x, axis=-1):
    x = x - x.max(axis=axis, keepdims=True)
    e = np.exp(x)
    return e / e.sum(axis=axis, keepdims=True)

X_reg = np.arcsinh(bulk_profile_matrix[:, top_k_idx])
y_reg = log_frac_matrix

train_idx_reg, test_idx_reg = train_test_split(
    np.arange(n_bulks),
    test_size=REGRESSION_TEST_FRACTION,
    random_state=RANDOM_SEED,
)

enet = MultiOutputRegressor(
    Pipeline([
        ('scaler', StandardScaler()),
        ('enet', ElasticNet(alpha=0.01, l1_ratio=0.5, random_state=RANDOM_SEED)),
    ])
)
enet.fit(X_reg[train_idx_reg], y_reg[train_idx_reg])

rf_reg = RandomForestRegressor(n_estimators=300, random_state=RANDOM_SEED)
rf_reg.fit(X_reg[train_idx_reg], y_reg[train_idx_reg])

for idx in test_idx_reg:
    rec = pseudobulk_records[idx]
    x = X_reg[idx:idx + 1]
    rec['enet_fractions'] = softmax(enet.predict(x))[0]
    rec['rf_reg_fractions'] = softmax(rf_reg.predict(x))[0]

print(f'Trained ElasticNet + RandomForestRegressor on {len(train_idx_reg)} pseudo-bulks, evaluating on {len(test_idx_reg)} held out')
example_rec = pseudobulk_records[test_idx_reg[0]]
print('Example (first held-out pseudo-bulk):')
print('  true:  ', dict(zip(classes, example_rec['true_fractions'].round(3))))
print('  enet:  ', dict(zip(classes, example_rec['enet_fractions'].round(3))))
print('  rf_reg:', dict(zip(classes, example_rec['rf_reg_fractions'].round(3))))

## 10. Combine Results

In [ ]:
rows = []
for rec in pseudobulk_records:
    row = {'bulk_id': rec['bulk_id']}
    for i, c in enumerate(classes):
        row[f'true_{c}'] = rec['true_fractions'][i]
        row[f'nnls_{c}'] = rec['nnls_fractions'][i]
        row[f'sc_{c}'] = rec['sc_fractions'][i]
        row[f'enet_{c}'] = rec['enet_fractions'][i] if 'enet_fractions' in rec else np.nan
        row[f'rf_reg_{c}'] = rec['rf_reg_fractions'][i] if 'rf_reg_fractions' in rec else np.nan
    rows.append(row)

results_df = pd.DataFrame(rows)
results_df.to_csv('pseudobulk_deconvolution_results.csv', index=False)
results_df.head()

## 11. Evaluate: True Fraction vs. Each Estimate, and the Two Estimates Against Each Other

Focused on the `fused` class since that's the fraction of specific interest, but the per-class
summary table below covers all three.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

panels = [
    (axes[0], results_df['true_fused'], results_df['nnls_fused'], 'C0',
     'True fused fraction', 'NNLS-estimated fused fraction', 'True vs. NNLS'),
    (axes[1], results_df['true_fused'], results_df['sc_fused'], 'darkorange',
     'True fused fraction', 'Single-cell-aggregated fused fraction', 'True vs. Single-Cell Classifier'),
    (axes[2], results_df['nnls_fused'], results_df['sc_fused'], 'seagreen',
     'NNLS-estimated fused fraction', 'Single-cell-aggregated fused fraction', 'NNLS vs. Single-Cell Classifier'),
]

for ax, x, y, color, xlabel, ylabel, title in panels:
    ax.scatter(x, y, alpha=0.6, s=15, color=color)
    ax.plot([0, 1], [0, 1], 'k--', linewidth=1)
    r, _ = pearsonr(x, y)
    ax.text(
        0.05, 0.95, f'$R^2$ = {r**2:.3f}\n$r$ = {r:.3f}',
        transform=ax.transAxes, ha='left', va='top', fontsize=10,
        bbox=dict(boxstyle='round', facecolor='white', alpha=0.7, edgecolor='none'),
    )
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.set_title(title)

for ax in axes:
    ax.set_xlim(-0.02, 1.02)
    ax.set_ylim(-0.02, 1.02)

plt.tight_layout()
plt.savefig('pseudobulk_fused_fraction_comparison.png')
plt.show()

## 12. Evaluate the Regression-Based Estimators (ElasticNet / RF) vs. Truth

Same R²/r annotation style as above, restricted to the held-out test pseudo-bulks (the only ones these two models never trained on).

In [ ]:
plot_df = results_df[results_df['enet_fused'].notna()]

fig, axes = plt.subplots(1, 2, figsize=(10, 5))

panels = [
    (axes[0], plot_df['true_fused'], plot_df['enet_fused'], 'mediumpurple',
     'True fused fraction', 'ElasticNet-estimated fused fraction', 'True vs. ElasticNet'),
    (axes[1], plot_df['true_fused'], plot_df['rf_reg_fused'], 'crimson',
     'True fused fraction', 'RF-regression-estimated fused fraction', 'True vs. RF Regression'),
]

for ax, x, y, color, xlabel, ylabel, title in panels:
    ax.scatter(x, y, alpha=0.6, s=15, color=color)
    ax.plot([0, 1], [0, 1], 'k--', linewidth=1)
    r, _ = pearsonr(x, y)
    ax.text(
        0.05, 0.95, f'$R^2$ = {r**2:.3f}\n$r$ = {r:.3f}',
        transform=ax.transAxes, ha='left', va='top', fontsize=10,
        bbox=dict(boxstyle='round', facecolor='white', alpha=0.7, edgecolor='none'),
    )
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.set_title(title)

for ax in axes:
    ax.set_xlim(-0.02, 1.02)
    ax.set_ylim(-0.02, 1.02)

plt.tight_layout()
plt.savefig('pseudobulk_fused_fraction_regression_comparison.png')
plt.show()

## 13. Summary Table Across All Methods

In [ ]:
summary_rows = []
for c in classes:
    true_c = results_df[f'true_{c}']
    row = {'class': c}
    for method in ['nnls', 'sc', 'enet', 'rf_reg']:
        est_c = results_df[f'{method}_{c}']
        mask = est_c.notna()
        r, _ = pearsonr(true_c[mask], est_c[mask])
        row[f'{method}_n'] = int(mask.sum())
        row[f'{method}_mae'] = (true_c[mask] - est_c[mask]).abs().mean()
        row[f'{method}_pearson_r'] = r
    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv('pseudobulk_deconvolution_summary.csv', index=False)
summary_df